# Kiểm tra nhanh Whisper Large với video tiếng Việt

Notebook lấy một đoạn ngắn từ một video trong Google Drive, chạy nhận dạng giọng nói và hiển thị kết quả để đánh giá chất lượng trước khi xử lý toàn bộ dữ liệu.

In [ ]:
!nvidia-smi
!pip -q install -U transformers accelerate safetensors
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Cấu hình kiểm tra
from pathlib import Path

DATASET_DIRECTORY = Path('/content/drive/MyDrive/AI Challenge/Dataset_Directory')
TEST_FOLDER = "Videos_L29_a/video" #@param {type:"string"}
TEST_VIDEO_NAME = "L29_V001.mp4" #@param {type:"string"}
TEST_START_SECONDS = 0 #@param {type:"integer"}
TEST_DURATION_SECONDS = 60 #@param {type:"integer"}
MODEL_ID = 'vinai/PhoWhisper-large'

dataset_root = DATASET_DIRECTORY.resolve()
test_folder_path = (DATASET_DIRECTORY / TEST_FOLDER).resolve()
assert dataset_root in test_folder_path.parents, 'Thư mục test phải nằm trong Dataset_Directory'
assert test_folder_path.is_dir(), f'Không tìm thấy thư mục: {test_folder_path}'
VIDEO_EXTENSIONS = {'.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v', '.mpeg', '.mpg'}
if TEST_VIDEO_NAME.strip():
    video_path = (test_folder_path / TEST_VIDEO_NAME.strip()).resolve()
    assert test_folder_path in video_path.parents, 'Video phải nằm trong thư mục test'
    assert video_path.is_file(), f'Không tìm thấy video: {video_path}'
else:
    test_videos = sorted(
        p for p in test_folder_path.rglob('*')
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    )
    assert test_videos, f'Không tìm thấy video trong: {test_folder_path}'
    video_path = test_videos[0]
assert TEST_START_SECONDS >= 0, 'Thời điểm bắt đầu không được âm'
assert TEST_DURATION_SECONDS > 0, 'Thời lượng kiểm tra phải lớn hơn 0'

clip_path = Path('/content/whisper_test_clip.wav')
print('Video:', video_path)
print(f'Đoạn kiểm tra: từ {TEST_START_SECONDS}s, dài {TEST_DURATION_SECONDS}s')

In [ ]:
import subprocess

subprocess.run([
    'ffmpeg', '-y', '-ss', str(TEST_START_SECONDS), '-i', str(video_path),
    '-t', str(TEST_DURATION_SECONDS), '-vn', '-ac', '1', '-ar', '16000',
    str(clip_path),
], check=True)
print('Đã tạo đoạn âm thanh kiểm tra:', clip_path)

In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

assert torch.cuda.is_available(), 'Hãy bật GPU: Runtime → Change runtime type → GPU'
print('GPU:', torch.cuda.get_device_name(0))
torch_dtype = torch.float16
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_ID, dtype=torch_dtype, low_cpu_mem_usage=True,
).to('cuda:0')
processor = AutoProcessor.from_pretrained(MODEL_ID)
transcriber = pipeline(
    'automatic-speech-recognition',
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device='cuda:0',
)
result = transcriber(
    str(clip_path),
    return_timestamps=True,
)

def format_timestamp(seconds):
    if seconds is None:
        return '??:??:??.???'
    milliseconds = round(float(seconds) * 1000)
    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    secs, millis = divmod(remainder, 1_000)
    return f'{hours:02d}:{minutes:02d}:{secs:02d}.{millis:03d}'

# Timestamp do Whisper trả về tính từ đầu clip WAV. Cộng offset để ra thời gian thật trong video.
timestamped_segments = []
for chunk in result.get('chunks', []):
    clip_start, clip_end = chunk.get('timestamp', (None, None))
    video_start = TEST_START_SECONDS + clip_start if clip_start is not None else None
    video_end = TEST_START_SECONDS + clip_end if clip_end is not None else None
    timestamped_segments.append({
        'text': chunk.get('text', '').strip(),
        'clip_start': clip_start,
        'clip_end': clip_end,
        'video_start': video_start,
        'video_end': video_end,
    })

print('\n--- KẾT QUẢ CÓ TIMESTAMP TRONG VIDEO ---\n')
for segment in timestamped_segments:
    print(
        f"[{format_timestamp(segment['video_start'])} --> "
        f"{format_timestamp(segment['video_end'])}] {segment['text']}"
    )
if not timestamped_segments:
    print(result.get('text', '').strip())

In [ ]:
import json

output_directory = Path('/content/drive/MyDrive/AI Challenge/Transcripts/_quick_test')
output_directory.mkdir(parents=True, exist_ok=True)
output_stem = output_directory / f'{video_path.stem}_{TEST_START_SECONDS}s_{TEST_DURATION_SECONDS}s'
saved_clip_path = output_stem.with_suffix('.wav')
saved_clip_path.write_bytes(clip_path.read_bytes())
timestamped_text = '\n'.join(
    f"[{format_timestamp(segment['video_start'])} --> {format_timestamp(segment['video_end'])}] {segment['text']}"
    for segment in timestamped_segments
)
output_stem.with_suffix('.txt').write_text(
    (timestamped_text or result.get('text', '').strip()) + '\n', encoding='utf-8'
)
output_payload = {
    'video_id': video_path.stem,
    'test_start_seconds': TEST_START_SECONDS,
    'test_duration_seconds': TEST_DURATION_SECONDS,
    'text': result.get('text', '').strip(),
    'segments': timestamped_segments,
}
output_stem.with_suffix('.json').write_text(
    json.dumps(output_payload, ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Đã lưu âm thanh so sánh:', saved_clip_path)
print('Đã lưu kết quả test:', output_stem)